# jax.vmap

In [1]:
import jax
import jax.numpy as jnp

## Basic Usage

`jax.vmap(f)` transforms a function written for a single example into one that operates over a batch — without writing explicit loops.

Under the hood it uses vectorized hardware instructions, so it is both cleaner and faster than a Python `for` loop.

In [2]:
# written for a single vector
def l2_norm(x):
    return jnp.sqrt(jnp.sum(x ** 2))

batch = jnp.array([[1.0, 0.0], [3.0, 4.0], [0.0, 1.0]])

# naive approach: Python loop
print([float(l2_norm(batch[i])) for i in range(len(batch))])

# vmap approach: no loop, fully vectorized
batched_l2 = jax.vmap(l2_norm)
print(batched_l2(batch))   # [1., 5., 1.]

[1.0, 5.0, 1.0]
[1. 5. 1.]


## in_axes

`in_axes` controls which axis of each argument is the batch axis.
- `0` (default): batch along the first axis
- `None`: the argument is shared across the batch (not mapped)
- `1`: batch along the second axis

In [3]:
def scale(x, factor):
    return x * factor

xs = jnp.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0]])  # (3, 2) — batch of vectors
f  = 2.0                                                 # scalar shared across batch

# map over xs (axis 0), broadcast f to every element (None = not mapped)
batched_scale = jax.vmap(scale, in_axes=(0, None))
print(batched_scale(xs, f))

[[ 2.  4.]
 [ 6.  8.]
 [10. 12.]]


## out_axes

`out_axes` controls which axis of the output holds the batch dimension. Defaults to `0`.

In [4]:
def double(x):
    return x * 2

xs = jnp.array([[1.0, 2.0, 3.0],
                [4.0, 5.0, 6.0]])   # shape (2, 3)

# default: batch axis goes to axis 0 of output  =>  shape (2, 3)
print(jax.vmap(double, out_axes=0)(xs).shape)

# out_axes=1: batch axis goes to axis 1 of output  =>  shape (3, 2)
print(jax.vmap(double, out_axes=1)(xs).shape)

(2, 3)
(3, 2)


## Combining vmap and grad

`vmap` and `grad` compose cleanly. A common pattern is to compute per-sample gradients — something that would otherwise require a loop.

In [5]:
def loss_single(w, x, y):
    # MSE loss for one sample
    pred = jnp.dot(w, x)
    return (pred - y) ** 2

w  = jnp.array([1.0, 2.0])
xs = jnp.array([[1.0, 0.0], [0.0, 1.0], [1.0, 1.0]])  # 3 samples
ys = jnp.array([1.0, 2.0, 3.0])

# per-sample gradient of loss w.r.t. w
per_sample_grads = jax.vmap(jax.grad(loss_single), in_axes=(None, 0, 0))
print(per_sample_grads(w, xs, ys))  # shape (3, 2)

[[0. 0.]
 [0. 0.]
 [0. 0.]]


## Batched Matrix-Vector Multiply

A practical example: applying a single weight matrix to a batch of vectors. Without `vmap` you would need `jnp.einsum` or an explicit batch dimension — `vmap` lets you keep the single-sample signature.

In [6]:
def linear(W, x):
    return W @ x   # written for a single vector x

W  = jnp.ones((4, 3))
xs = jnp.ones((8, 3))   # batch of 8 vectors

# map over x (axis 0), keep W shared
batched_linear = jax.vmap(linear, in_axes=(None, 0))
out = batched_linear(W, xs)
print(out.shape)   # (8, 4)

# equivalent (but less readable) without vmap
print(jnp.einsum("ij,bj->bi", W, xs).shape)

(8, 4)
(8, 4)


## Nested vmap

`vmap` can be nested to map over multiple independent batch dimensions.

In [7]:
def dot(x, y):
    return jnp.dot(x, y)

xs = jnp.ones((3, 4))   # 3 vectors of dim 4
ys = jnp.ones((5, 4))   # 5 vectors of dim 4

# compute all 3×5 pairwise dot products
pairwise_dot = jax.vmap(jax.vmap(dot, in_axes=(None, 0)), in_axes=(0, None))
result = pairwise_dot(xs, ys)
print(result.shape)   # (3, 5)

(3, 5)
